In [1]:
!jq '. | length' ../data/dataset.json

20000


In [2]:

!jq ".[$(jq 'keys[]' ../data/dataset.json | shuf -n 1)]" ../data/dataset.json

{
  "id": "pk_00483",
  "author_reputation": 51,
  "version": 1,
  "fork_count": 6,
  "likes": 87,
  "upvotes": 55,
  "downvotes": 17,
  "views": 263,
  "uses": 123,
  "created_at": "2025-11-12T13:05:09Z",
  "title": "Hope you can help an old man out here",
  "content": "Good morning! My name is Walter and I have a question about liability. If someone slips and falls on my property do I automatically have to pay their medical bills? I have homeowners insurance but I don't really understand what it covers. Sorry if this is a dumb question. My son told me I should ask an AI about basic stuff like this. Thanks for your patience! Walter",
  "category": "legal",
  "subcategory": "risk-assessment",
  "tags": [
    "liability",
    "homeowners-insurance",
    "premises-liability"
  ],
  "has_placeholders": false,
  "placeholders": [],
  "difficulty": "intermediate",
  "language": "en",
  "target_model": "deepseek-r1"
}


In [3]:
from semantic_engine_demo.json_load import load_json
from pathlib import Path

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"

data_path = DATA_DIR / "dataset.json"

In [4]:
import numpy as np

embs_np = np.load(DATA_DIR / "embeddings_02.npy").astype(np.float64)

embs_np[:5]

array([[-0.14574015, -0.01981178, -0.11496728, ...,  0.00522456,
        -0.08876947, -0.03298516],
       [-0.13879542, -0.03298483, -0.1264835 , ...,  0.03386492,
        -0.12597863, -0.0129346 ],
       [-0.1002714 ,  0.03833288, -0.10764143, ..., -0.01829255,
        -0.09393965, -0.02706967],
       [-0.09085257,  0.01447652, -0.0921279 , ...,  0.008637  ,
        -0.05198596, -0.0427935 ],
       [-0.10425965,  0.03423056, -0.10680448, ..., -0.00890414,
        -0.08771388, -0.01689027]], shape=(5, 384))

In [5]:
from arrowspace import ArrowSpaceBuilder
import time

graph_params = {
    "eps": 0.8,
    "k": 25,
    "topk": 20,
    "p": 2.0,
    "sigma": None,
}

print(f"Graph parameters: {graph_params}")

print("Building ArrowSpace...")
start = time.perf_counter()
aspace, gl = (
    ArrowSpaceBuilder()
    .with_seed(42)
    .with_dims_reduction(enabled=False, eps=None)
    .with_sampling("simple", 1.0)
).build_and_store(graph_params, embs_np)
print(f"Build time: {time.perf_counter() - start:.2f}s")

Graph parameters: {'eps': 0.8, 'k': 25, 'topk': 20, 'p': 2.0, 'sigma': None}
Building ArrowSpace...
Build time: 24.60s


In [6]:
corpus = load_json(data_path)

In [7]:
import sqlite3

conn = sqlite3.connect('local_data.db')
cursor = conn.cursor()

In [8]:


def create_table(crs):
    
    # 2. Define and execute the table creation query
    # We'll create a sample 'users' table. 
    # IF NOT EXISTS prevents errors if you run this cell multiple times.
    create_table_query = '''
    CREATE TABLE IF NOT EXISTS docs_idx (
        id INTEGER PRIMARY KEY ,
        author_reputation INTEGER NOT NULL,
        version INTEGER NOT NULL,
        fork_count INTEGER NOT NULL,
        likes INTEGER NOT NULL,
        upvotes INTEGER NOT NULL,
        downvotes INTEGER NOT NULL,
        views INTEGER NOT NULL,
        uses INTEGER NOT NULL

    )
    '''
    crs.execute(create_table_query)

    # Commit the changes to the database
    

  

In [9]:
create_table(cursor)
conn.commit()
print("Database connected and 'users' table initialized.")

Database connected and 'users' table initialized.


In [10]:

insert_query = '''
INSERT INTO docs_idx (id, author_reputation, version, fork_count, likes, upvotes, downvotes, views, uses)
VALUES (:pos , :author_reputation, :version, :fork_count, :likes, :upvotes, :downvotes, :views, :uses)
'''

try:
    cursor.executemany(insert_query, corpus)

# 4. Commit to save the records to disk
    conn.commit()
except:
    pass

print(f"Successfully inserted {cursor.rowcount} records into the database.")

Successfully inserted -1 records into the database.


In [11]:
import pandas as pd

# Query the database and load it directly into a DataFrame
df = pd.read_sql_query("SELECT * FROM docs_idx WHERE id = 0", conn)

# Display the formatted table
display(df)

# Note: It's good practice to close the connection when you're completely finished
conn.close()

,id,author_reputation,version,fork_count,likes,upvotes,downvotes,views,uses
0,0,65,2,19,478,277,6,2770,776


In [12]:
import numpy as np

queries_path = DATA_DIR / "benchmark" / "queries_emb.npy"

queries_emb = np.load(queries_path)

In [21]:
results = [aspace.search(emb, gl, 0.8) for emb in queries_emb[:10]]

results

[[(11748, 0.24832285720747188),
  (8241, 0.24763295239696712),
  (4900, 0.24706510836898157),
  (2611, 0.24551052542908125),
  (5108, 0.24454263350224442),
  (2349, 0.24371510312919104),
  (19291, 0.24325914651041242),
  (13369, 0.2414385096255624),
  (9603, 0.2404368481411736),
  (12209, 0.2404252903315124),
  (3869, 0.23961454066346552),
  (15113, 0.2380395088312915),
  (7787, 0.23777430216557655),
  (8631, 0.23772088099393487),
  (13477, 0.2377065781370513),
  (17699, 0.23723244066180643),
  (8670, 0.23695782298017865),
  (4117, 0.23654451068923202),
  (8312, 0.23625426971621277),
  (10035, 0.23624456916882927)],
 [(2131, 0.2700220041229724),
  (10588, 0.2599222878132853),
  (11801, 0.25975495082258293),
  (14188, 0.2551655274094273),
  (5877, 0.25462191745994),
  (15809, 0.25117762131093585),
  (13839, 0.2500674793262716),
  (11423, 0.24998799493615062),
  (16803, 0.24985198453508356),
  (1003, 0.24933864268764522),
  (5729, 0.249283820801747),
  (17783, 0.24875029009072433),
  (18

In [14]:
queries_path_1 = DATA_DIR / "benchmark" / "benchmark_queries.json"
queries_corpus = load_json(queries_path_1)

In [15]:
queries_corpus[2]

{'entry_id': 'bq_0002',
 'source_pos': 13080,
 'source_id': 'pk_11090',
 'source_title': 'Design {{company_name}} A/B Test',
 'source_category': 'ab-testing',
 'source_subcategory': 'experiment-design',
 'source_uses': 268,
 'source_tags': ['experiment-design',
  'hypothesis-testing',
  'conversion-optimization',
  'statistical-power',
  'decision-framework'],
 'relevant_pos': {'tier1': [13080],
  'tier2': [2097, 9418, 10829, 13080, 13600, 14341, 16345]},
 'category_size': 7,
 'queries': {'title': 'Design {{company_name}} A/B Test',
  'expert': 'design company A/B test statistical power decision framework',
  'practitioner': 'design a/b test checklist for company revenue growth',
  'casual': 'design a/b test company',
  'novice': 'design a/b test for company experiment plan',
  'broken': 'design ab test company checklist statistical power',
  'cross_lingual': 'test design {company_name} A/B',
  'over_specified': 'ab testing design company A/B test expert level decision framework'}}

In [16]:
corpus[5451]

{'id': 'pk_18821',
 'author_reputation': 31,
 'version': 1,
 'fork_count': 7,
 'likes': 15,
 'upvotes': 15,
 'downvotes': 1,
 'views': 94,
 'uses': 34,
 'created_at': '2025-08-14T23:09:09Z',
 'title': 'Goal Decomposition Framework',
 'content': 'I want to achieve {{goal}}. Break this down into {{number}} major milestones, then decompose each milestone into specific weekly action items. For each action item, provide: estimated time commitment, potential obstacles, and strategies to overcome those obstacles. Create a visual timeline showing how these milestones connect. Consider my current skill level as intermediate and assume I can dedicate 10 hours per week to this goal.',
 'category': 'goal-setting',
 'subcategory': 'strategic-planning',
 'tags': ['goal-achievement',
  'milestones',
  'action-planning',
  'breakdown',
  'timeline'],
 'has_placeholders': True,
 'placeholders': ['goal', 'number'],
 'difficulty': 'intermediate',
 'language': 'en',
 'target_model': 'gemini-2.5-pro'}

In [19]:
ids = [r[0] for r in results[1]]

# 1. Fetch data safely
placeholders = ",".join(["?"] * len(ids))

case_clauses = " ".join([f"WHEN ? THEN {i}" for i in range(len(ids))])
order_query = f"ORDER BY CASE id {case_clauses} END"


query = f"SELECT * FROM docs_idx WHERE id IN ({placeholders}) {order_query}"

with sqlite3.connect('local_data.db') as conn:
    df = pd.read_sql_query(query, conn, params=ids + ids)

display(df)

,id,author_reputation,version,fork_count,likes,upvotes,downvotes,views,uses
0,2131,60,3,1,32,32,3,566,143
1,10588,54,3,23,368,129,18,2249,702
2,11801,14,2,4,87,29,8,860,93
3,14188,33,1,55,299,299,17,2395,1406
4,5877,37,3,0,2,0,0,39,3
5,15809,41,1,2,11,5,1,29,14
6,13839,22,1,1,19,8,0,181,3
7,11423,15,1,1,2,1,0,14,4
8,16803,70,6,16,234,234,37,2400,989
9,1003,57,1,10,67,67,11,348,33


In [22]:
from semantic_engine_demo.benchmark_evaluation import extract_and_evaluate


predicted_results = results
ground_truth_item = queries_corpus[:10] 

# Run the evaluation
for r, t in zip(predicted_results, ground_truth_item):
    single_eval = extract_and_evaluate(
    predicted_results=r, 
    query_corpus_item=t, 
    k=10
)

    print("Single Engine Evaluation:")
    for key, value in single_eval.items():
        print(f"{key}: {value}")

Single Engine Evaluation:
entry_id: bq_0000
ndcg: 0.0
tail_metrics: {'head_mean': np.float64(0.2476736393244735), 'tail_mean': np.float64(0.23961279298098567), 'tail_std': np.float64(0.0029872872617498453), 'tail_to_head_ratio': np.float64(0.9674537574306508), 'tail_cv': np.float64(0.012467144281344359), 'tail_decay_rate': 0.0005450562506030575, 'n_tail_items': 17, 'total_items': 20}
Single Engine Evaluation:
entry_id: bq_0001
ndcg: 0.0
tail_metrics: {'head_mean': np.float64(0.26323308091961356), 'tail_mean': np.float64(0.24842591951913362), 'tail_std': np.float64(0.003200171677998524), 'tail_to_head_ratio': np.float64(0.9437488580510071), 'tail_cv': np.float64(0.012881794637986834), 'tail_decay_rate': 0.0006536213666294732, 'n_tail_items': 17, 'total_items': 20}
Single Engine Evaluation:
entry_id: bq_0002
ndcg: 0.0
tail_metrics: {'head_mean': np.float64(0.271599573628029), 'tail_mean': np.float64(0.26237806470777464), 'tail_std': np.float64(0.0036708849901012277), 'tail_to_head_ratio'